# 🚀 The Ultimate NumPy Masterclass: From Zero to Machine Learning
> **A Comprehensive, User-Friendly, and Interactive Reference Guide**
> 
> *Target Audience:* Beginners looking for crystal-clear clarity, Data Scientists needing a fast reference, and ML Engineers building vector algorithms.

---

### 🌟 How to Use This Notebook
1. **Interactive & Modular:** Each concept is broken down into small, digestible chunks with clear explanations, visual diagrams, and focused code cells.
2. **Visual Mental Models:** Pay special attention to the ASCII diagrams illustrating memory layouts, multidimensional slicing, axes, and broadcasting rules.
3. **Pre-Executed & Verified:** All outputs are pre-rendered so you can read this like an illustrated book, or run each cell interactively to experiment.
4. **Quick Navigation:** Use the **Table of Contents** below or each chapter's `[⬆️ Back to Table of Contents]` link to jump smoothly between topics.

---

<a id="table-of-contents"></a>
## 📑 Table of Contents

| Part | Chapter | Key Topics Covered |
| :---: | :--- | :--- |
| **I** | [1. Architecture & Why NumPy is Fast](#ch1) | Python Lists vs ndarrays, C-Contiguity, Memory Benchmarks, Vectorization |
| **I** | [2. Array Creation Methods](#ch2) | `np.array`, `zeros`, `ones`, `full`, `arange` vs `linspace`, `logspace`, matrices |
| **I** | [3. Array Anatomy, Attributes & Memory](#ch3) | `shape`, `ndim`, `dtype`, `itemsize`, `nbytes`, Strides, and Memory Flags |
| **I** | [4. Modern Random Number Generation](#ch4) | `default_rng()`, Uniform, Normal, Integers, Sampling, and Shuffling |
| **II** | [5. Indexing, Slicing & Striding](#ch5) | 1D & 2D slicing grids, negative indexing, step stride, ellipsis `...`, `newaxis` |
| **II** | [6. Views vs Copies: Memory Mechanics](#ch6) | Data buffer sharing, `arr.base`, accidental mutations, `.copy()` best practices |
| **II** | [7. Advanced Indexing](#ch7) | Boolean masking, Bitwise operators (`&`, `|`, `~`), `np.where`, Fancy indexing |
| **II** | [8. Reshaping & Structural Transforms](#ch8) | `reshape(-1)`, `flatten()` vs `ravel()`, Transpose `.T`, `expand_dims`, `squeeze` |
| **III** | [9. Joining, Stacking, Splitting & Repeating](#ch9) | `concatenate`, `vstack`, `hstack`, `stack`, `split`, `tile` vs `repeat` |
| **III** | [10. Universal Functions (ufuncs)](#ch10) | Vectorized math, trigonometry, `reduce`, `accumulate`, `outer` |
| **III** | [11. Broadcasting Mechanics: The 3 Rules](#ch11) | Shape alignment rules, dimension stretching, practical column normalization |
| **III** | [12. Aggregations, Reductions & The Axis Model](#ch12) | Mental model for `axis=0` vs `axis=1`, `keepdims=True`, `nan`-safe functions |
| **IV** | [13. Sorting, Searching & Partitioning](#ch13) | `sort` vs `argsort`, fast $O(N)$ top-K selection with `argpartition`, `unique` |
| **IV** | [14. Linear Algebra Mastery (`np.linalg`)](#ch14) | `@` vs `*`, `det`, `inv`, `pinv`, solving systems $Ax=b$, norms, SVD, eigenvalues |
| **IV** | [15. Machine Learning Vectorization in Action](#ch15) | Pairwise Euclidean distance, Stable Softmax, One-Hot Encoding, MSE, Normal Eq |
| **IV** | [16. Optimization, Memory & Anti-Patterns](#ch16) | Preallocation vs `np.append` anti-pattern, C-order vs Fortran-order cache |
| **V** | [Quick Reference Cheat Sheet](#summary) | High-frequency function summary table and companion practice exercise links |

<a id="ch1"></a>
---
# Part I: Foundations & Memory Architecture

## 1. Architecture & Why NumPy is Fast

### 1.1 Python Lists vs. NumPy ndarrays: The Memory Layout
To understand why NumPy is 50x–100x faster than standard Python, we must look at how system RAM stores both structures:

```
[Standard Python List]: Array of Pointers to Heap PyObjects
List Header ---> [ Pointer 0 ] ---> PyObject (type, refcount, value = 42)
                 [ Pointer 1 ] ---> PyObject (type, refcount, value = 18)
                 [ Pointer 2 ] ---> PyObject (type, refcount, value = 99)
  * Values scattered across RAM -> Massive CPU Cache Misses
  * Type-checking & dynamic dispatch on EVERY iteration
  * High memory overhead (approx 28+ bytes per integer!)

[NumPy ndarray]: One Contiguous C-Style Memory Buffer
Array Header ---> [ 42 | 18 | 99 | 74 | 12 | ... ] (raw bytes in continuous memory)
  * Sequential RAM -> Perfect CPU L1/L2/L3 cache prefetching
  * Fixed, uniform dtype -> Direct pointer arithmetic: address = base + index * 8
  * Releases Python's GIL during C-loops -> True hardware parallelism
```

In [1]:
import numpy as np
import sys
import time

print(f"NumPy Version: {np.__version__}")

NumPy Version: 2.5.3


### 1.2 Memory Footprint Benchmark
Let's measure the actual memory consumed by a Python list of 10,000 integers versus a NumPy array of 10,000 integers.

In [2]:
# Create 10,000 integers in Python list vs NumPy array
n_items = 10_000
py_list = list(range(n_items))
np_arr = np.arange(n_items, dtype=np.int64)

# Memory of the list container + memory of each integer PyObject
py_list_bytes = sys.getsizeof(py_list) + sum(sys.getsizeof(x) for x in py_list)
# Memory of the NumPy array buffer + array header
np_arr_bytes = np_arr.nbytes + sys.getsizeof(np_arr)

print(f"📦 Python List Memory:  {py_list_bytes:>8,} bytes")
print(f"⚡ NumPy Array Memory:   {np_arr_bytes:>8,} bytes")
print(f"🎉 NumPy is {py_list_bytes / np_arr_bytes:.1f}x more memory-compact!")

📦 Python List Memory:   360,056 bytes
⚡ NumPy Array Memory:    160,112 bytes
🎉 NumPy is 2.2x more memory-compact!


### 1.3 Execution Speed Benchmark: Vectorization vs Python Loop
Now let's compute element-wise addition on 2 arrays of **5,000,000 numbers**.

In [3]:
N = 5_000_000
list_a = list(range(N))
list_b = list(range(N))
arr_a = np.arange(N, dtype=np.float64)
arr_b = np.arange(N, dtype=np.float64)

# 1. Pure Python Loop (benchmarked on 500k and scaled for timing responsiveness)
t0 = time.perf_counter()
_ = [a + b for a, b in zip(list_a[:500_000], list_b[:500_000])]
t1 = time.perf_counter()
py_time = (t1 - t0) * 10 # Extrapolated to 5M

# 2. NumPy Vectorized C-Speed Addition
t0 = time.perf_counter()
_ = arr_a + arr_b
t1 = time.perf_counter()
np_time = t1 - t0

print(f"🐢 Python Loop (extrapolated 5M): {py_time:.4f} seconds")
print(f"🚀 NumPy Vectorized (5M):         {np_time:.4f} seconds")
print(f"⚡ Speedup:                       {py_time / np_time:.1f}x faster!")

🐢 Python Loop (extrapolated 5M): 0.3534 seconds
🚀 NumPy Vectorized (5M):         0.0196 seconds
⚡ Speedup:                       18.0x faster!


> 💡 **Key Takeaway:**
> Never write `for` loops in Python to do elementwise math on arrays. NumPy's vectorization delegates the entire loop down to optimized, pre-compiled C/Fortran SIMD vector instructions on your CPU!
>
> [⬆️ Back to Table of Contents](#table-of-contents)

<a id="ch2"></a>
---
## 2. Array Creation Methods

NumPy provides built-in constructors to create arrays of any shape, dimension, and datatype without writing loops.

### 2.1 From Python Sequences (`np.array`)
You can convert any nested Python list or tuple into an `ndarray`. Specifying `dtype` prevents unwanted type coercions.

In [4]:
# 1D Array with explicit 32-bit float
a_1d = np.array([10, 20, 30, 40], dtype=np.float32)

# 2D Matrix (2 rows x 3 columns)
a_2d = np.array([[1, 2, 3], [4, 5, 6]], dtype=np.int32)

print("1D Array (float32):", a_1d)
print("2D Matrix (2x3):\n", a_2d)

1D Array (float32): [10. 20. 30. 40.]
2D Matrix (2x3):
 [[1 2 3]
 [4 5 6]]


### 2.2 Constant Initializers: Zeros, Ones, Full, Empty
These are used to pre-allocate memory buffers before running computations.

In [5]:
# Zeros array of shape (2, 4)
zeros = np.zeros(shape=(2, 4), dtype=np.float64)

# Ones tensor of shape (2, 2, 3)
ones = np.ones(shape=(2, 2, 3), dtype=np.int16)

# Fill with a custom scalar value (e.g., 99.0)
filled = np.full(shape=(2, 3), fill_value=99.0)

# Empty allocates UNINITIALIZED memory (fastest when you immediately overwrite it)
empty = np.empty(shape=(2, 2))

print("Zeros (2x4):\n", zeros)
print("\nFull with 99.0 (2x3):\n", filled)

Zeros (2x4):
 [[0. 0. 0. 0.]
 [0. 0. 0. 0.]]

Full with 99.0 (2x3):
 [[99. 99. 99.]
 [99. 99. 99.]]


### 2.3 Like-Constructors: Matching Shape & Dtype
When you have an existing array and want another array with identical shape and data type without manually typing the shape:

In [6]:
template = np.array([[1.5, 2.5, 3.5], [4.5, 5.5, 6.5]])

zeros_like = np.zeros_like(template)
full_like = np.full_like(template, fill_value=-1.0)

print("Template shape:", template.shape, "dtype:", template.dtype)
print("Zeros like template:\n", zeros_like)
print("Full like template:\n", full_like)

Template shape: (2, 3) dtype: float64
Zeros like template:
 [[0. 0. 0.]
 [0. 0. 0.]]
Full like template:
 [[-1. -1. -1.]
 [-1. -1. -1.]]


### 2.4 Numerical Ranges: `arange` vs `linspace`
This is one of the most critical distinctions in scientific Python:

| Function | Signature | Use When... | End-point Included? |
| :--- | :--- | :--- | :---: |
| `np.arange(start, stop, step)` | Specify **step size** | You care about the step interval | ❌ No (exclusive) |
| `np.linspace(start, stop, num)` | Specify **number of points** | You care about exact sample count | ✅ Yes (inclusive) |

In [7]:
# arange: step is 2 (stop is exclusive)
r_arange = np.arange(start=0, stop=10, step=2)

# linspace: exactly 5 points evenly spaced between 0 and 1 (inclusive)
r_linspace = np.linspace(start=0.0, stop=1.0, num=5)

# logspace: 4 points logarithmically spaced from 10^1 to 10^4
r_logspace = np.logspace(start=1, stop=4, num=4, base=10.0)

print("np.arange(0, 10, 2):     ", r_arange)
print("np.linspace(0, 1, num=5):", r_linspace)
print("np.logspace(1, 4, num=4):", r_logspace)

np.arange(0, 10, 2):      [0 2 4 6 8]
np.linspace(0, 1, num=5): [0.   0.25 0.5  0.75 1.  ]
np.logspace(1, 4, num=4): [   10.   100.  1000. 10000.]


### 2.5 Matrix & Diagonal Constructors
Special initializers commonly used in Linear Algebra and Machine Learning:

In [8]:
# 3x3 Identity matrix (1s on main diagonal, 0s elsewhere)
identity = np.eye(3)

# Extract diagonal from matrix OR construct matrix from diagonal elements
diag_matrix = np.diag([10, 20, 30])

# Upper and Lower triangular portions
mat = np.ones((3, 3), dtype=int)
upper = np.triu(mat)
lower = np.tril(mat)

print("Identity Matrix (3x3):\n", identity)
print("\nDiagonal Matrix:\n", diag_matrix)
print("\nUpper Triangle:\n", upper)

Identity Matrix (3x3):
 [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]

Diagonal Matrix:
 [[10  0  0]
 [ 0 20  0]
 [ 0  0 30]]

Upper Triangle:
 [[1 1 1]
 [0 1 1]
 [0 0 1]]


> 💡 **Pro-Tip:**
> Avoid using `np.arange` with floating-point steps like `np.arange(0, 1, 0.1)` because floating-point rounding errors can cause unexpected array lengths! Always prefer **`np.linspace(0, 1, 11)`** for float ranges.
>
> [⬆️ Back to Table of Contents](#table-of-contents)

<a id="ch3"></a>
---
## 3. Array Anatomy, Attributes & Memory

Every NumPy array consists of two things:
1. **The Data Buffer**: Raw bytes stored in contiguous memory.
2. **The Array Header / Metadata**: Lightweight Python object storing the `shape`, `dtype`, `strides`, and memory flags.

```
Array Header:
  shape   : (3, 4)       -> 3 rows, 4 columns
  dtype   : int32        -> 4 bytes per element
  strides : (16, 4)      -> Step 16 bytes for next row, 4 bytes for next col
  data    : 0x10a8040    -> Pointer to raw byte buffer in RAM
```

In [9]:
# Create a 3x4 array of 32-bit integers (4 bytes each)
arr = np.arange(12, dtype=np.int32).reshape(3, 4)

print("--- Array Inspection ---")
print("Array:\n", arr)
print(f"ndim (dimensions):       {arr.ndim}")
print(f"shape (rows, cols):      {arr.shape}")
print(f"size (total elements):   {arr.size}")
print(f"dtype (data type):       {arr.dtype}")
print(f"itemsize (bytes/item):   {arr.itemsize} bytes")
print(f"nbytes (total memory):   {arr.nbytes} bytes ({arr.size} * {arr.itemsize})")

--- Array Inspection ---
Array:
 [[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
ndim (dimensions):       2
shape (rows, cols):      (3, 4)
size (total elements):   12
dtype (data type):       int32
itemsize (bytes/item):   4 bytes
nbytes (total memory):   48 bytes (12 * 4)


### 3.2 Demystifying `strides`
**Strides** tell NumPy how many bytes the CPU must step through memory to advance by 1 index in each dimension:
- To advance to the next column: step **4 bytes** (size of 1 `int32`).
- To advance to the next row: step 4 items $\times$ 4 bytes = **16 bytes**.

In [10]:
print("Strides tuple (row_bytes, col_bytes):", arr.strides)

# Inspect memory flags: is it C-contiguous? Does it own its memory buffer?
print("\nMemory Flags:")
print("C-Contiguous (Row-major):", arr.flags['C_CONTIGUOUS'])
print("Fortran (Column-major):  ", arr.flags['F_CONTIGUOUS'])
print("Owns its memory buffer:  ", arr.flags['OWNDATA'])

Strides tuple (row_bytes, col_bytes): (16, 4)

Memory Flags:
C-Contiguous (Row-major): True
Fortran (Column-major):   False
Owns its memory buffer:   False


> 🧠 **Under the Hood:**
> When you transpose an array (`arr.T`), NumPy **does not copy any data**! It simply swaps the strides from `(16, 4)` to `(4, 16)`. This is why transposing a 10 GB array in NumPy takes $0.000001$ seconds!
>
> [⬆️ Back to Table of Contents](#table-of-contents)

<a id="ch4"></a>
---
## 4. Modern Random Number Generation (`np.random.default_rng`)

In older NumPy code, you will see `np.random.seed(42)` and `np.random.rand()`.
**NumPy 1.17+ deprecated this legacy API.**
The modern standard is the **Generator API** initialized with `np.random.default_rng()`, which uses the high-performance **PCG64** pseudo-random number generator.

In [11]:
# Initialize modern generator (optionally pass a seed for exact reproducibility)
rng = np.random.default_rng(seed=42)

# 1. Random floats in [0.0, 1.0)
floats = rng.random(size=(2, 3))

# 2. Random integers between low (inclusive) and high (exclusive)
# endpoint=True makes high inclusive
ints = rng.integers(low=1, high=100, size=(2, 3))

# 3. Standard Normal distribution (mean=0, variance=1)
std_norm = rng.standard_normal(size=(2, 3))

# 4. Custom Normal distribution: mean (loc)=10.0, std dev (scale)=2.5
custom_norm = rng.normal(loc=10.0, scale=2.5, size=4)

print("Floats [0, 1):\n", floats.round(3))
print("\nIntegers [1, 100):\n", ints)
print("\nStandard Normal (Gaussian):\n", std_norm.round(3))
print("\nNormal(mu=10, sigma=2.5):\n", custom_norm.round(2))

Floats [0, 1):
 [[0.774 0.439 0.859]
 [0.697 0.094 0.976]]

Integers [1, 100):
 [[73 76 72]
 [78 51 13]]

Standard Normal (Gaussian):
 [[-0.853  0.879  0.778]
 [ 0.066  1.127  0.468]]

Normal(mu=10, sigma=2.5):
 [ 7.85 10.92  7.6  12.2 ]


### 4.2 Shuffling and Random Sampling
Used extensively for train/test data splitting, stochastic sampling, and Monte Carlo simulations:

In [12]:
deck = np.array(["Ace", "King", "Queen", "Jack", "10", "9", "8"])

# 1. rng.choice: Randomly pick elements with or without replacement
drawn = rng.choice(deck, size=3, replace=False)
print("Random pick 3 cards (no replacement):", drawn)

# 2. rng.permutation: Returns a newly shuffled COPY
permuted = rng.permutation(deck)
print("Permuted copy:  ", permuted)

# 3. rng.shuffle: Shuffles an array IN-PLACE (mutates original)
arr_to_shuffle = np.arange(10)
rng.shuffle(arr_to_shuffle)
print("In-place shuffled array:", arr_to_shuffle)

Random pick 3 cards (no replacement): ['King' 'Jack' '8']
Permuted copy:   ['Queen' '9' 'Ace' 'Jack' '10' 'King' '8']
In-place shuffled array: [6 3 8 2 9 5 1 7 4 0]


> 💡 **Best Practice:**
> Always pass a seed like `rng = np.random.default_rng(seed=42)` when sharing notebooks or training ML models so your results are 100% reproducible for others!
>
> [⬆️ Back to Table of Contents](#table-of-contents)

<a id="ch5"></a>
---
# Part II: Indexing, Slicing & Views

## 5. Indexing, Slicing & Multi-Dimensional Striding

### 5.1 1D Array Slicing: `[start:stop:step]`
```
Positive Index:     0    1    2    3    4    5    6    7
Values:          [ 10,  20,  30,  40,  50,  60,  70,  80 ]
Negative Index:    -8   -7   -6   -5   -4   -3   -2   -1
```

In [13]:
x = np.arange(10, 90, 10) # [10, 20, 30, 40, 50, 60, 70, 80]

print("Original array:         ", x)
print("Element at index 2:     ", x[2])       # 30
print("Element at index -1:    ", x[-1])      # 80 (last item)
print("Slice [1:5]:            ", x[1:5])     # [20, 30, 40, 50]
print("Slice [:4] (first 4):   ", x[:4])      # [10, 20, 30, 40]
print("Slice [::2] (every 2nd):", x[::2])     # [10, 30, 50, 70]
print("Reverse array [::-1]:   ", x[::-1])    # Reversed!

Original array:          [10 20 30 40 50 60 70 80]
Element at index 2:      30
Element at index -1:     80
Slice [1:5]:             [20 30 40 50]
Slice [:4] (first 4):    [10 20 30 40]
Slice [::2] (every 2nd): [10 30 50 70]
Reverse array [::-1]:    [80 70 60 50 40 30 20 10]


### 5.2 2D Matrix Indexing & Slicing: `[row_slice, col_slice]`
Unlike Python lists where you use chained brackets `mat[row][col]`, NumPy uses a clean comma syntax: **`mat[row_idx, col_idx]`**.

```
Matrix indices:
           Col 0    Col 1    Col 2    Col 3
  Row 0 [   0,       1,       2,       3   ]
  Row 1 [   4,       5,       6,       7   ]
  Row 2 [   8,       9,      10,      11   ]
```

In [14]:
mat = np.arange(12).reshape(3, 4)
print("Matrix (3x4):\n", mat)

# Single element at Row 1, Col 2
print("\nElement at [1, 2]:      ", mat[1, 2])

# Entire Row 1
print("Entire Row 1:           ", mat[1, :])

# Entire Column 2 (returns 1D array)
print("Entire Column 2:        ", mat[:, 2])

# Submatrix: Rows 0..1, Cols 1..3
print("Subgrid (Rows 0-1, Cols 1-3):\n", mat[0:2, 1:4])

Matrix (3x4):
 [[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]

Element at [1, 2]:       6
Entire Row 1:            [4 5 6 7]
Entire Column 2:         [ 2  6 10]
Subgrid (Rows 0-1, Cols 1-3):
 [[1 2 3]
 [5 6 7]]


### 5.3 3D Tensor Slicing & The Ellipsis (`...`) Syntax
In deep learning, data tensors often have 4 dimensions: `(batch_size, channels, height, width)`.
The **Ellipsis (`...`)** represents "all intervening axes":

In [15]:
# 3D Tensor: 2 batches, 3 rows, 4 columns
tensor = np.arange(24).reshape(2, 3, 4)

# Grab all batches, all rows, but only column 0:
# The following two are completely identical:
slice_colon = tensor[:, :, 0]
slice_ellipsis = tensor[..., 0]

print("Slice using ellipsis tensor[..., 0]:\n", slice_ellipsis)
print("Are both slices identical?", np.array_equal(slice_colon, slice_ellipsis))

Slice using ellipsis tensor[..., 0]:
 [[ 0  4  8]
 [12 16 20]]
Are both slices identical? True


### 5.4 Adding New Axes (`np.newaxis` / `None`)
Adding dimensions is critical for vector alignment in Machine Learning:

In [16]:
v = np.array([1, 2, 3]) # 1D vector of shape (3,)

row_vector = v[np.newaxis, :]  # Shape: (1, 3)
col_vector = v[:, None]         # Shape: (3, 1)

print("Original shape:  ", v.shape)
print("Row vector shape:", row_vector.shape)
print("Col vector shape:", col_vector.shape)
print("Col vector content:\n", col_vector)

Original shape:   (3,)
Row vector shape: (1, 3)
Col vector shape: (3, 1)
Col vector content:
 [[1]
 [2]
 [3]]


> 💡 **Key Takeaway:**
> Use `arr[:, None]` or `np.newaxis` whenever you need to promote a 1D vector into a 2D column matrix for matrix multiplication or broadcasting.
>
> [⬆️ Back to Table of Contents](#table-of-contents)

<a id="ch6"></a>
---
## 6. Views vs Copies: Memory Mechanics & Pitfalls

This is the **#1 source of silent bugs** in data science and NumPy programming!

```
[View]: Slicing DOES NOT copy data!
original_arr ---> [ 10 | 20 | 30 | 40 | 50 ] <--- slice_view
Modifying slice_view directly modifies original_arr!

[Copy]: Explicit .copy() creates an independent memory buffer
original_arr ---> [ 10 | 20 | 30 | 40 | 50 ]
copy_arr     ---> [ 10 | 20 | 30 | 40 | 50 ] (separate RAM)
Modifying copy_arr leaves original_arr safe and untouched.
```

In [17]:
original = np.array([10, 20, 30, 40, 50])

# Slicing creates a VIEW (shares memory!)
view_slice = original[1:4]
print("Before modification:")
print("Original:", original)
print("View:    ", view_slice)

# Modify the view!
view_slice[0] = 999

print("\nAfter modifying view_slice[0] = 999:")
print("View:    ", view_slice)
print("Original:", original, " <-- ORIGINAL WAS MUTATED!")

Before modification:
Original: [10 20 30 40 50]
View:     [20 30 40]

After modifying view_slice[0] = 999:
View:     [999  30  40]
Original: [ 10 999  30  40  50]  <-- ORIGINAL WAS MUTATED!


### 6.2 Checking Memory Ownership with `.base`
How do you know if an array is a view or owns its memory?

In [18]:
# If .base is None -> The array owns its memory
# If .base is another array -> It is a VIEW pointing to that array
print("original.base is None:  ", original.base is None)
print("view_slice.base is original:", view_slice.base is original)

# To create a completely safe, independent copy:
safe_copy = original[1:4].copy()
safe_copy[0] = -777

print("\nAfter modifying safe_copy[0] = -777:")
print("safe_copy:", safe_copy)
print("original: ", original, " <-- UNTOUCHED!")

original.base is None:   True
view_slice.base is original: True

After modifying safe_copy[0] = -777:
safe_copy: [-777   30   40]
original:  [ 10 999  30  40  50]  <-- UNTOUCHED!


> ⚠️ **Common Bug Alert:**
> In standard Python, `my_list[1:4]` creates a brand-new copy. In NumPy, `arr[1:4]` creates a **VIEW**! If you don't want modifications to propagate back to your data, always append **`.copy()`**!
>
> [⬆️ Back to Table of Contents](#table-of-contents)

<a id="ch7"></a>
---
## 7. Advanced Indexing (Boolean Masks & Integer Arrays)

### 7.1 Boolean Masking: Filtering Elements by Condition
When you apply a logical comparison to an array, NumPy evaluates it element-wise and returns a boolean array of identical shape:

In [19]:
scores = np.array([55, 88, 42, 95, 73, 61, 30, 99])

# Create a boolean mask: which students passed (score >= 60)?
mask = scores >= 60
print("Boolean Mask:\n", mask)

# Pass the mask back into scores to extract only matching elements:
passed_students = scores[mask]
print("Passing scores:\n", passed_students)

Boolean Mask:
 [False  True False  True  True  True False  True]
Passing scores:
 [88 95 73 61 99]


### 7.2 Combining Conditions with Bitwise Operators (`&`, `|`, `~`)
⚠️ **Critical Python Pitfall:**
You **cannot** use Python's `and`, `or`, `not` keywords with NumPy arrays!
You **must** use bitwise operators:
- `&` for AND
- `|` for OR
- `~` for NOT
- **Each condition MUST be wrapped in parentheses `()`!**

In [20]:
# Find scores between 70 AND 95 (inclusive)
filtered = scores[(scores >= 70) & (scores <= 95)]
print("Scores in [70, 95]:", filtered)

# Invert mask using ~ (NOT): find scores that failed (< 60)
failed = scores[~(scores >= 60)]
print("Failed scores:    ", failed)

Scores in [70, 95]: [88 95 73]
Failed scores:     [55 42 30]


### 7.3 Vectorized Conditional Assignment: `np.where`
`np.where(condition, value_if_true, value_if_false)` is the vectorized equivalent of the ternary operator `cond ? a : b`.

In [21]:
# Label scores as 'Pass' or 'Fail'
status = np.where(scores >= 60, "Pass", "Fail")
print("Scores: ", scores)
print("Status: ", status)

# Cap outlier values: if score > 90, set to 90, else keep original
capped = np.where(scores > 90, 90, scores)
print("Capped: ", capped)

Scores:  [55 88 42 95 73 61 30 99]
Status:  ['Fail' 'Pass' 'Fail' 'Pass' 'Pass' 'Pass' 'Fail' 'Pass']
Capped:  [55 88 42 90 73 61 30 90]


### 7.4 Fancy (Integer Array) Indexing
Fancy indexing means passing a list or array of specific indices to extract elements at arbitrary positions.
> 📌 **Key Rule:** While slicing returns a view, **Fancy Indexing ALWAYS returns a deep copy!**

In [22]:
arr = np.array([100, 200, 300, 400, 500, 600, 700])

# Select indices 0, 3, and 6
indices = [0, 3, 6]
subset = arr[indices]

print("Selected items [0, 3, 6]:", subset)
print("Does subset own its memory?", subset.base is None)

Selected items [0, 3, 6]: [100 400 700]
Does subset own its memory? True


> 💡 **Summary of Slicing vs Fancy Indexing:**
> - `arr[1:5]` -> **View** (fast, zero memory copied, edits affect parent).
> - `arr[[1, 2, 4]]` -> **Copy** (allocates new memory, edits do not affect parent).
>
> [⬆️ Back to Table of Contents](#table-of-contents)

<a id="ch8"></a>
---
# Part III: Transformations & Array Math

## 8. Array Reshaping & Structural Transformations

### 8.1 Reshaping & The `-1` Automatic Dimension Trick
You can reshape an array into any dimension as long as the total number of elements (`size`) remains unchanged.
Passing **`-1`** tells NumPy: *"calculate this dimension automatically for me!"*

In [23]:
a = np.arange(12) # 1D array of 12 numbers

# Reshape into 3 rows, 4 columns
mat_3x4 = a.reshape(3, 4)

# Reshape into 2 rows, let NumPy infer the columns (-1 -> 6)
mat_2x6 = a.reshape(2, -1)

# Reshape into 3D tensor: 2 batches x 2 rows x 3 columns
tensor_3d = a.reshape(2, 2, 3)

print("Original 1D:  ", a)
print("Reshaped (3x4):\n", mat_3x4)
print("\nReshaped (2, -1) -> shape:", mat_2x6.shape)
print("Tensor 3D shape:", tensor_3d.shape)

Original 1D:   [ 0  1  2  3  4  5  6  7  8  9 10 11]
Reshaped (3x4):
 [[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]

Reshaped (2, -1) -> shape: (2, 6)
Tensor 3D shape: (2, 2, 3)


### 8.2 Flattening: `flatten()` vs `ravel()`
When collapsing an nD matrix down to a 1D vector:

| Method | Returns | Performance | Modifying Output Mutates Original? |
| :--- | :---: | :---: | :---: |
| `arr.flatten()` | **Deep Copy** | Slower (allocates new RAM) | ❌ No |
| `arr.ravel()` | **View** (when possible) | Instant ($O(1)$ memory) | ⚠️ Yes |

In [24]:
mat = np.array([[1, 2], [3, 4]])

flat = mat.flatten() # Copy
rav = mat.ravel()    # View

print("Original:\n", mat)
print("flatten():", flat)
print("ravel():  ", rav)
print("rav.base is mat?", rav.base is mat)

Original:
 [[1 2]
 [3 4]]
flatten(): [1 2 3 4]
ravel():   [1 2 3 4]
rav.base is mat? True


### 8.3 Transpose (`.T`) and Dimension Squeezing
- `.T` or `np.transpose()` reverses the order of axes.
- `np.squeeze()` strips away dimensions of length 1.
- `np.expand_dims()` injects a dimension of length 1.

In [25]:
mat = np.array([[1, 2, 3], [4, 5, 6]])
print("Original (2x3):\n", mat)
print("Transposed (3x2):\n", mat.T)

# Dummy 4D array with unit dimensions: (1, 3, 1, 4)
dummy = np.ones((1, 3, 1, 4))
squeezed = np.squeeze(dummy)
print("\nBefore squeeze:", dummy.shape)
print("After squeeze: ", squeezed.shape)

Original (2x3):
 [[1 2 3]
 [4 5 6]]
Transposed (3x2):
 [[1 4]
 [2 5]
 [3 6]]

Before squeeze: (1, 3, 1, 4)
After squeeze:  (3, 4)


> [⬆️ Back to Table of Contents](#table-of-contents)

<a id="ch9"></a>
---
## 9. Joining, Stacking, Splitting & Repeating Arrays

### 9.1 Concatenation & Stacking
NumPy gives you flexible ways to combine arrays:
- `np.concatenate([a, b], axis=...)`: joins along an **existing** axis.
- `np.vstack([a, b])`: stacks vertically (row-wise, increases rows).
- `np.hstack([a, b])`: stacks horizontally (col-wise, increases columns).
- `np.stack([a, b], axis=...)`: stacks along a **brand new** axis.

In [26]:
a = np.array([[1, 2], [3, 4]])
b = np.array([[5, 6], [7, 8]])

# Vertical stack (axis=0)
v_stacked = np.vstack([a, b])

# Horizontal stack (axis=1)
h_stacked = np.hstack([a, b])

# New axis stack (creates 3D array: 2 x 2 x 2)
new_stacked = np.stack([a, b], axis=0)

print("Array A:\n", a)
print("\nArray B:\n", b)
print("\nVertical Stack (vstack):\n", v_stacked)
print("\nHorizontal Stack (hstack):\n", h_stacked)
print("\nNew Axis Stack (shape):\n", new_stacked.shape)

Array A:
 [[1 2]
 [3 4]]

Array B:
 [[5 6]
 [7 8]]

Vertical Stack (vstack):
 [[1 2]
 [3 4]
 [5 6]
 [7 8]]

Horizontal Stack (hstack):
 [[1 2 5 6]
 [3 4 7 8]]

New Axis Stack (shape):
 (2, 2, 2)


### 9.2 Splitting Arrays
Opposite of concatenation: dividing an array into sub-arrays.

In [27]:
x = np.arange(12).reshape(3, 4)
print("Original (3x4):\n", x)

# Split horizontally into 2 equal parts of (3, 2)
left, right = np.hsplit(x, 2)
print("\nLeft half:\n", left)
print("\nRight half:\n", right)

Original (3x4):
 [[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]

Left half:
 [[0 1]
 [4 5]
 [8 9]]

Right half:
 [[ 2  3]
 [ 6  7]
 [10 11]]


### 9.3 Tiling vs Repeating
- `np.repeat(arr, reps)`: repeats **individual elements** sequentially.
- `np.tile(arr, reps)`: repeats the **entire array structure like tiles on a floor**.

In [28]:
base = np.array([1, 2, 3])

repeated = np.repeat(base, repeats=3)
tiled = np.tile(base, reps=3)

print("Base:   ", base)
print("repeat: ", repeated)
print("tile:   ", tiled)

Base:    [1 2 3]
repeat:  [1 1 1 2 2 2 3 3 3]
tile:    [1 2 3 1 2 3 1 2 3]


> [⬆️ Back to Table of Contents](#table-of-contents)

<a id="ch10"></a>
---
## 10. Universal Functions (ufuncs) & Elementwise Math

A **ufunc** (Universal Function) is a compiled C-function that operates element-wise on arrays with SIMD hardware optimization.

### 10.1 Standard Elementwise Math
All standard arithmetic operators map directly to fast ufuncs:

In [29]:
x = np.array([1.0, 4.0, 9.0, 16.0])

print("Original:       ", x)
print("x + 5:          ", x + 5)
print("sqrt(x):        ", np.sqrt(x))
print("exp(x):         ", np.exp(np.array([1, 2, 3])).round(3))
print("log(x):         ", np.log(x).round(3))
print("Power x ** 2:   ", x ** 2)
print("Clip to [3, 10]:", np.clip(x, 3.0, 10.0))

Original:        [ 1.  4.  9. 16.]
x + 5:           [ 6.  9. 14. 21.]
sqrt(x):         [1. 2. 3. 4.]
exp(x):          [ 2.718  7.389 20.086]
log(x):          [0.    1.386 2.197 2.773]
Power x ** 2:    [  1.  16.  81. 256.]
Clip to [3, 10]: [ 3.  4.  9. 10.]


### 10.2 Special ufunc Methods: `reduce`, `accumulate`, `outer`
Every binary ufunc (like `add`, `multiply`) has built-in methods that eliminate the need for loops:

In [30]:
nums = np.array([1, 2, 3, 4, 5])

# 1. reduce: repeatedly applies operation across all items (sum / prod)
sum_result = np.add.reduce(nums)        # 1 + 2 + 3 + 4 + 5 = 15
prod_result = np.multiply.reduce(nums)  # 1 * 2 * 3 * 4 * 5 = 120

# 2. accumulate: running cumulative sum / running product
running_sum = np.add.accumulate(nums)

# 3. outer: computes outer product table for all pairs
outer_table = np.multiply.outer(nums, nums)

print("Sum reduce:     ", sum_result)
print("Product reduce: ", prod_result)
print("Running cumsum: ", running_sum)
print("Outer multiplication table (5x5):\n", outer_table)

Sum reduce:      15
Product reduce:  120
Running cumsum:  [ 1  3  6 10 15]
Outer multiplication table (5x5):
 [[ 1  2  3  4  5]
 [ 2  4  6  8 10]
 [ 3  6  9 12 15]
 [ 4  8 12 16 20]
 [ 5 10 15 20 25]]


> [⬆️ Back to Table of Contents](#table-of-contents)

<a id="ch11"></a>
---
## 11. Broadcasting Mechanics: The 3 Golden Rules

**Broadcasting** allows NumPy to perform arithmetic operations on arrays of **different shapes** without copying any data into memory.

### 📐 The 3 Golden Rules of Broadcasting:
1. **Rule 1 (Align Right):** Compare shapes from right to left (trailing dimensions first). If the arrays have a different number of dimensions, prepend `1`s to the shorter shape until dimensions match.
2. **Rule 2 (Compatibility):** Two dimensions are compatible if:
   - They are **equal**, OR
   - One of them is **1**.
3. **Rule 3 (Stretching):** Dimensions of size `1` are virtually stretched (via stride 0) to match the larger size.

```
Example 1: Matrix (3, 3) + Row Vector (3,)
  Matrix:     (3, 3)
  Vector:        (3,)  --> Prepended to (1, 3)
  Broadcast:  (3, 3) + (1, 3) -> Row is virtually replicated 3 times!

Example 2: Column Vector (3, 1) + Row Vector (1, 4)
  Col Vector: (3, 1)  --> Stretched horizontally to (3, 4)
  Row Vector: (1, 4)  --> Stretched vertically to (3, 4)
  Result:     (3, 4)  --> 2D grid matrix!
```

In [31]:
# Case 1: 2D Matrix (3, 3) + 1D Row Vector (3,)
matrix = np.ones((3, 3))
row_vec = np.array([10, 20, 30])

result_1 = matrix + row_vec
print("Matrix (3x3):\n", matrix)
print("\nRow Vector (3,):\n", row_vec)
print("\nBroadcast Result (3x3):\n", result_1)

Matrix (3x3):
 [[1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]]

Row Vector (3,):
 [10 20 30]

Broadcast Result (3x3):
 [[11. 21. 31.]
 [11. 21. 31.]
 [11. 21. 31.]]


### 11.2 Outer Grid Broadcasting: `(3, 1) + (1, 4) -> (3, 4)`

In [32]:
col = np.array([[10], [20], [30]]) # Shape (3, 1)
row = np.array([[1, 2, 3, 4]])      # Shape (1, 4)

grid = col + row
print("Col shape:", col.shape)
print("Row shape:", row.shape)
print("Grid result shape:", grid.shape)
print("Result matrix:\n", grid)

Col shape: (3, 1)
Row shape: (1, 4)
Grid result shape: (3, 4)
Result matrix:
 [[11 12 13 14]
 [21 22 23 24]
 [31 32 33 34]]


### 11.3 Practical ML Example: Feature Standardization (Z-score normalization)
Suppose you have a feature matrix `X` (100 samples $\times$ 3 features). How do you normalize each feature column to have mean 0 and std 1 without a loop?

In [33]:
# Simulated dataset: 4 samples, 3 features
X = np.array([
    [10.0, 200.0, 0.5],
    [20.0, 150.0, 0.7],
    [30.0, 300.0, 0.2],
    [40.0, 250.0, 0.9]
])

# Compute mean and std for each column (axis=0)
col_mean = X.mean(axis=0) # Shape: (3,)
col_std = X.std(axis=0)   # Shape: (3,)

# Broadcasting: (4, 3) - (3,) -> Automatically broadcasts down rows!
X_normalized = (X - col_mean) / col_std

print("Original Features (4x3):\n", X)
print("\nColumn Means (3,):", col_mean.round(2))
print("\nNormalized Features (zero-mean, unit-variance):\n", X_normalized.round(2))

Original Features (4x3):
 [[1.0e+01 2.0e+02 5.0e-01]
 [2.0e+01 1.5e+02 7.0e-01]
 [3.0e+01 3.0e+02 2.0e-01]
 [4.0e+01 2.5e+02 9.0e-01]]

Column Means (3,): [ 25.   225.     0.57]

Normalized Features (zero-mean, unit-variance):
 [[-1.34 -0.45 -0.29]
 [-0.45 -1.34  0.48]
 [ 0.45  1.34 -1.45]
 [ 1.34  0.45  1.26]]


> ⚠️ **Gotcha / Incompatible Shapes Error:**
> If you try to add `(3, 3) + (2,)`, NumPy will raise `ValueError: operands could not be broadcast together`. Always ensure the trailing dimensions match or one of them is 1!
>
> [⬆️ Back to Table of Contents](#table-of-contents)

<a id="ch12"></a>
---
## 12. Aggregations, Reductions & The Axis Mental Model

### 🧭 The Axis Mental Model
The `#1` point of confusion for beginners is what `axis=0` vs `axis=1` means.

```
                    axis = 1  (Across columns / along each row) --->
                    Col 0     Col 1     Col 2
         Row 0  [    10,       20,       30    ]  ---> sum across row: 60
axis = 0 Row 1  [    40,       50,       60    ]  ---> sum across row: 150
(Down)   Row 2  [    70,       80,       90    ]  ---> sum across row: 240
   |
   V  sum down cols: 120       150       180
```
- **`axis=0` collapses rows**: computes down each column (result has length = number of columns).
- **`axis=1` collapses columns**: computes across each row (result has length = number of rows).
- **`axis=None` (default)**: collapses the entire array to a single scalar value.

In [34]:
mat = np.array([
    [10, 20, 30],
    [40, 50, 60],
    [70, 80, 90]
])

print("Overall sum (all elements):", mat.sum())
print("axis=0 sum (down columns):  ", mat.sum(axis=0))
print("axis=1 sum (across rows):   ", mat.sum(axis=1))

Overall sum (all elements): 450
axis=0 sum (down columns):   [120 150 180]
axis=1 sum (across rows):    [ 60 150 240]


### 12.2 The Power of `keepdims=True`
When you aggregate an axis, NumPy removes that dimension by default:
- `(3, 3).mean(axis=1)` -> shape `(3,)` (1D vector)
If you want to broadcast this back onto the original matrix, shape `(3,)` will fail or align incorrectly.
By adding **`keepdims=True`**, NumPy keeps the collapsed dimension with size 1:
- `(3, 3).mean(axis=1, keepdims=True)` -> shape `(3, 1)` (ready for broadcasting!)

In [35]:
row_means_no_keep = mat.mean(axis=1)
row_means_keep = mat.mean(axis=1, keepdims=True)

print("Without keepdims shape:", row_means_no_keep.shape)
print("With keepdims shape:   ", row_means_keep.shape)

# Subtract row mean from each row smoothly:
demeaned_rows = mat - row_means_keep
print("\nRow-demeaned matrix:\n", demeaned_rows)

Without keepdims shape: (3,)
With keepdims shape:    (3, 1)

Row-demeaned matrix:
 [[-10.   0.  10.]
 [-10.   0.  10.]
 [-10.   0.  10.]]


### 12.3 Summary Statistics & NaN-Handling
NumPy includes fast statistical functions, including `nan`-safe variants that ignore missing values:

In [36]:
data = np.array([10.0, 25.0, np.nan, 40.0, 100.0])

print("Standard mean with NaN: ", np.mean(data))      # Produces nan!
print("Safe np.nanmean:         ", np.nanmean(data))   # Correctly computes 43.75!
print("Safe np.nansum:          ", np.nansum(data))

# Finding min/max indices
clean_arr = np.array([42, 11, 89, 23, 7, 56])
print("\nMin value:", clean_arr.min(), "at index (argmin):", clean_arr.argmin())
print("Max value:", clean_arr.max(), "at index (argmax):", clean_arr.argmax())
print("Median:   ", np.median(clean_arr))
print("75th Percentile:", np.percentile(clean_arr, 75))

Standard mean with NaN:  nan
Safe np.nanmean:          43.75
Safe np.nansum:           175.0

Min value: 7 at index (argmin): 4
Max value: 89 at index (argmax): 2
Median:    32.5
75th Percentile: 52.5


> [⬆️ Back to Table of Contents](#table-of-contents)

<a id="ch13"></a>
---
# Part IV: Advanced Operations & Linear Algebra

## 13. Sorting, Searching & Partitioning

### 13.1 `sort` vs `argsort`
- `np.sort(arr)`: returns a sorted **copy** of values.
- `np.argsort(arr)`: returns the **indices** that would sort the array.
`argsort` is essential in machine learning for ranking predictions, nearest neighbors, and sorting parallel arrays.

In [37]:
names = np.array(["Alice", "Bob", "Charlie", "David", "Eve"])
scores = np.array([78, 95, 62, 88, 71])

# Get indices that sort scores ascending
sorted_indices = np.argsort(scores)

print("Original scores:", scores)
print("Sorted indices: ", sorted_indices)
print("Scores sorted:  ", scores[sorted_indices])
print("Ranked names:   ", names[sorted_indices])

# For descending order (highest score first):
desc_indices = np.argsort(scores)[::-1]
print("Top students:   ", names[desc_indices])

Original scores: [78 95 62 88 71]
Sorted indices:  [2 4 0 3 1]
Scores sorted:   [62 71 78 88 95]
Ranked names:    ['Charlie' 'Eve' 'Alice' 'David' 'Bob']
Top students:    ['Bob' 'David' 'Alice' 'Eve' 'Charlie']


### 13.2 Fast $O(N)$ Top-K Selection: `np.partition` & `np.argpartition`
Sorting an array of 1,000,000 items takes $O(N \log N)$.
If you only need the **top 3 largest elements**, sorting the entire array is wasteful.
`np.partition` puts the $K$ smallest/largest elements in the first $K$ positions in **$O(N)$ linear time** without sorting everything!

In [38]:
large_data = np.array([42, 11, 89, 23, 7, 56, 99, 14, 3, 77])

# Put the 3 smallest elements at the beginning
partitioned = np.partition(large_data, kth=3)
print("Partitioned around kth=3:", partitioned)
print("3 smallest items:        ", partitioned[:3])

# Top 3 largest indices via argpartition
top3_idx = np.argpartition(large_data, kth=-3)[-3:]
print("Top 3 largest items:     ", large_data[top3_idx])

Partitioned around kth=3: [ 7  3 11 14 23 42 99 56 89 77]
3 smallest items:         [ 7  3 11]
Top 3 largest items:      [77 89 99]


### 13.3 Unique Values & Frequency Counts
Extract unique items and counts without converting to a Python dictionary:

In [39]:
labels = np.array(["cat", "dog", "cat", "bird", "dog", "cat", "dog"])

unique_vals, counts = np.unique(labels, return_counts=True)

print("Unique Classes:    ", unique_vals)
print("Class Frequencies: ", counts)

Unique Classes:     ['bird' 'cat' 'dog']
Class Frequencies:  [1 3 3]


> [⬆️ Back to Table of Contents](#table-of-contents)

<a id="ch14"></a>
---
## 14. Linear Algebra Mastery (`np.linalg`)

Linear algebra is the core mathematical engine behind Deep Learning, Computer Vision, and Machine Learning.

### 14.1 Elementwise `*` vs Matrix Multiplication `@`
```
Elementwise Multiplication (A * B): Multiplies corresponding items (Must be same shape)
[ a11  a12 ] * [ b11  b12 ] = [ a11*b11  a12*b12 ]
[ a21  a22 ]   [ b21  b22 ]   [ a21*b21  a22*b22 ]

Matrix Multiplication (A @ B): Row-by-Column Dot Product (M x K) @ (K x N) -> (M x N)
[ a11  a12 ] @ [ b11  b12 ] = [ a11*b11+a12*b21   a11*b12+a12*b22 ]
[ a21  a22 ]   [ b21  b22 ]   [ a21*b11+a22*b21   a21*b12+a22*b22 ]
```

In [40]:
A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])

# 1. Element-wise
print("Element-wise (A * B):\n", A * B)

# 2. Matrix Multiplication using @ operator
print("\nMatrix Multiplication (A @ B):\n", A @ B)

Element-wise (A * B):
 [[ 5 12]
 [21 32]]

Matrix Multiplication (A @ B):
 [[19 22]
 [43 50]]


### 14.2 Determinant, Trace, Rank, and Inverses

In [41]:
M = np.array([[4.0, 7.0], [2.0, 6.0]])

# Determinant: det(M) = 4*6 - 7*2 = 24 - 14 = 10
det = np.linalg.det(M)

# Trace: sum of diagonal elements (4 + 6 = 10)
trace = np.trace(M)

# Matrix Rank
rank = np.linalg.matrix_rank(M)

# Inverse: M^(-1) such that M @ M^(-1) = Identity
M_inv = np.linalg.inv(M)

print("Matrix M:\n", M)
print(f"Determinant: {det:.2f}")
print(f"Trace:       {trace}")
print(f"Matrix Rank: {rank}")
print("Inverse M^(-1):\n", M_inv)
print("Verify M @ M_inv is Identity:\n", np.round(M @ M_inv, 5))

Matrix M:
 [[4. 7.]
 [2. 6.]]
Determinant: 10.00
Trace:       10.0
Matrix Rank: 2
Inverse M^(-1):
 [[ 0.6 -0.7]
 [-0.2  0.4]]
Verify M @ M_inv is Identity:
 [[ 1. -0.]
 [-0.  1.]]


### 14.3 Solving Linear Systems ($Ax = b$)
Suppose you have the system of equations:
$$2x + y = 8$$
$$x + 3y = 14$$

> 💡 **Best Practice:** Never compute `np.linalg.inv(A) @ b` to solve a system.
> Use **`np.linalg.solve(A, b)`** instead. It uses LU decomposition with partial pivoting, which is significantly faster and numerically stable!

In [42]:
A = np.array([[2.0, 1.0], [1.0, 3.0]])
b = np.array([8.0, 14.0])

# Solve for x
solution = np.linalg.solve(A, b)

print("Solution [x, y]:", solution)
print("Verification A @ x == b:", np.allclose(A @ solution, b))

Solution [x, y]: [2. 4.]
Verification A @ x == b: True


### 14.4 Vector & Matrix Norms (`np.linalg.norm`)
Used for loss functions, regularization (L1 Lasso, L2 Ridge), and distance calculations:

In [43]:
v = np.array([3.0, -4.0])

# L1 Norm (Manhattan): |3| + |-4| = 7
norm_l1 = np.linalg.norm(v, ord=1)

# L2 Norm (Euclidean): sqrt(3^2 + (-4)^2) = 5
norm_l2 = np.linalg.norm(v, ord=2)

print("Vector:", v)
print("L1 Norm (Manhattan):", norm_l1)
print("L2 Norm (Euclidean):", norm_l2)

Vector: [ 3. -4.]
L1 Norm (Manhattan): 7.0
L2 Norm (Euclidean): 5.0


### 14.5 Eigenvalues & SVD (Principal Component Analysis)

In [44]:
# Symmetric covariance matrix
cov_mat = np.array([[2.0, 0.8], [0.8, 1.5]])

# Eigendecomposition
eigenvalues, eigenvectors = np.linalg.eigh(cov_mat)

# Singular Value Decomposition: M = U @ S @ Vt
U, S, Vt = np.linalg.svd(cov_mat)

print("Eigenvalues:  ", eigenvalues.round(3))
print("Eigenvectors:\n", eigenvectors.round(3))
print("Singular Values (S):", S.round(3))

Eigenvalues:   [0.912 2.588]
Eigenvectors:
 [[ 0.592 -0.806]
 [-0.806 -0.592]]
Singular Values (S): [2.588 0.912]


> [⬆️ Back to Table of Contents](#table-of-contents)

<a id="ch15"></a>
---
# Part V: Practical Machine Learning & Optimization

## 15. Practical Machine Learning Vectorization Building Blocks

In this chapter, we implement 5 essential ML algorithms completely vectorized without a single Python `for` loop.

### 15.1 Pairwise Euclidean Distance Matrix
Given a matrix $X$ of shape $(N, D)$ containing $N$ feature vectors, compute the $(N, N)$ distance matrix between all pairs of points.
**The Vectorized Trick:**
$$||a - b||^2 = ||a||^2 + ||b||^2 - 2 (a \cdot b)$$

In [45]:
# Simulated dataset: 5 points in 3D space
X = np.array([
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0],
    [7.0, 8.0, 9.0],
    [1.0, 0.0, 1.0],
    [2.0, 2.0, 2.0]
])

# Sum of squares along rows: shape (5, 1)
sq_norms = np.sum(X ** 2, axis=1, keepdims=True)

# Pairwise squared distance: ||a||^2 + ||b||^2 - 2(a @ b.T)
dists_sq = sq_norms + sq_norms.T - 2 * (X @ X.T)

# Fix any tiny negative floating point artifacts at 0
dists_sq = np.maximum(dists_sq, 0.0)
pairwise_dist = np.sqrt(dists_sq)

print("Pairwise Euclidean Distance Matrix (5x5):\n", pairwise_dist.round(2))

Pairwise Euclidean Distance Matrix (5x5):
 [[ 0.    5.2  10.39  2.83  1.41]
 [ 5.2   0.    5.2   7.68  5.39]
 [10.39  5.2   0.   12.81 10.49]
 [ 2.83  7.68 12.81  0.    2.45]
 [ 1.41  5.39 10.49  2.45  0.  ]]


### 15.2 Numerically Stable Softmax
The standard Softmax formula is:
$$\text{Softmax}(z_i) = \frac{e^{z_i}}{\sum_j e^{z_j}}$$
If $z_i = 1000$, $e^{1000}$ triggers **floating-point overflow (`inf`)** and returns `NaN`!
**The Stable Trick:** Subtract the row maximum $\max(z)$ before exponentiating:
$$\text{Softmax}(z_i) = \frac{e^{z_i - \max(z)}}{\sum_j e^{z_j - \max(z)}}$$

In [46]:
def stable_softmax(logits):
    # Subtract max per row for numerical stability
    shifted_logits = logits - np.max(logits, axis=-1, keepdims=True)
    exp_scores = np.exp(shifted_logits)
    probabilities = exp_scores / np.sum(exp_scores, axis=-1, keepdims=True)
    return probabilities

# Raw unnormalized model outputs for 2 samples across 3 classes
raw_logits = np.array([
    [1000.0, 1001.0, 1002.0], # Huge numbers that would normally overflow!
    [1.0, 2.0, 3.0]
])

probs = stable_softmax(raw_logits)
print("Raw Logits:\n", raw_logits)
print("\nSoftmax Probabilities:\n", probs.round(4))
print("Row sums verify to 1.0:", np.sum(probs, axis=-1))

Raw Logits:
 [[1.000e+03 1.001e+03 1.002e+03]
 [1.000e+00 2.000e+00 3.000e+00]]

Softmax Probabilities:
 [[0.09   0.2447 0.6652]
 [0.09   0.2447 0.6652]]
Row sums verify to 1.0: [1. 1.]


### 15.3 Vectorized One-Hot Encoding
Convert integer class labels into binary indicator vectors:

In [47]:
labels = np.array([0, 2, 1, 0, 3]) # 5 samples belonging to 4 classes
num_classes = 4

# One-line vectorized one-hot encoding using np.eye!
one_hot = np.eye(num_classes)[labels]

print("Class Labels:     ", labels)
print("One-Hot Encoded Matrix:\n", one_hot)

Class Labels:      [0 2 1 0 3]
One-Hot Encoded Matrix:
 [[1. 0. 0. 0.]
 [0. 0. 1. 0.]
 [0. 1. 0. 0.]
 [1. 0. 0. 0.]
 [0. 0. 0. 1.]]


### 15.4 Mean Squared Error (MSE) Loss Function

In [48]:
# Actual target values vs Model predictions
y_true = np.array([10.5, 14.2, 18.0, 22.1, 30.5])
y_pred = np.array([11.0, 13.8, 18.5, 21.0, 31.0])

# Vectorized MSE
mse = np.mean((y_true - y_pred) ** 2)
# Root Mean Squared Error (RMSE)
rmse = np.sqrt(mse)

print("True Targets:     ", y_true)
print("Predictions:      ", y_pred)
print(f"Mean Squared Error (MSE):  {mse:.4f}")
print(f"Root Mean Squared (RMSE):  {rmse:.4f}")

True Targets:      [10.5 14.2 18.  22.1 30.5]
Predictions:       [11.  13.8 18.5 21.  31. ]
Mean Squared Error (MSE):  0.4240
Root Mean Squared (RMSE):  0.6512


### 15.5 Closed-Form Linear Regression (The Normal Equation)
Given feature matrix $X$ and target vector $y$, the analytical solution that minimizes least squares error is:
$$\theta = (X^T X)^{-1} X^T y$$

In [49]:
# 1. Synthetic data: y = 2*x1 + 3*x2 + 5 + noise
rng = np.random.default_rng(42)
N = 100
X_raw = rng.uniform(0, 10, size=(N, 2))
true_weights = np.array([2.0, 3.0])
true_bias = 5.0
y = X_raw @ true_weights + true_bias + rng.normal(0, 0.1, size=N)

# 2. Add bias column (column of 1s) to X
X_design = np.hstack([np.ones((N, 1)), X_raw])

# 3. Solve Normal Equation: (X^T X)^(-1) X^T y
theta = np.linalg.inv(X_design.T @ X_design) @ X_design.T @ y

print("True Parameters:   Bias = 5.00,  Weights = [2.00, 3.00]")
print(f"Fitted Parameters: Bias = {theta[0]:.2f}, Weights = [{theta[1]:.2f}, {theta[2]:.2f}]")
print("Optimization converged with closed-form precision!")

True Parameters:   Bias = 5.00,  Weights = [2.00, 3.00]
Fitted Parameters: Bias = 4.97, Weights = [2.00, 3.00]
Optimization converged with closed-form precision!


> [⬆️ Back to Table of Contents](#table-of-contents)

<a id="ch16"></a>
---
## 16. Performance Optimization, Memory Layout & Anti-Patterns

### 16.1 The `#1` NumPy Anti-Pattern: `np.append()` Inside a Loop
In Python lists, `list.append()` runs in amortized $O(1)$ time.
In NumPy, `np.append()` **allocates a brand-new array and copies all existing data into it!**
Running `np.append()` in a loop of $N$ iterations runs in **$O(N^2)$ time** and destroys performance.

In [50]:
N = 25_000

# ❌ The Anti-Pattern: np.append inside loop
t0 = time.perf_counter()
bad_arr = np.array([])
for i in range(N):
    bad_arr = np.append(bad_arr, i)
t1 = time.perf_counter()
time_bad = t1 - t0

# ✅ The Best Practice: Preallocate memory once!
t0 = time.perf_counter()
good_arr = np.empty(N, dtype=np.float64)
for i in range(N):
    good_arr[i] = i
t1 = time.perf_counter()
time_good = t1 - t0

print(f"❌ np.append loop:   {time_bad:.4f} seconds")
print(f"✅ Preallocation:     {time_good:.4f} seconds")
print(f"⚡ Preallocation is  {time_bad / time_good:.1f}x faster!")

❌ np.append loop:   0.1219 seconds
✅ Preallocation:     0.0022 seconds
⚡ Preallocation is  55.5x faster!


### 16.2 Memory Layout: C-Order vs Fortran-Order Cache Performance
- **C-contiguous (Row-major, default):** Rows are stored adjacent in RAM. Summing across rows is lightning fast due to CPU cache line pre-fetching.
- **Fortran-contiguous (Column-major):** Columns are stored adjacent.

In [51]:
# Large matrix: 5,000 x 5,000 floats
mat = np.ones((5000, 5000), dtype=np.float64)

# 1. Sum along rows (matches C-contiguous layout in memory)
t0 = time.perf_counter()
_ = np.sum(mat, axis=1)
t1 = time.perf_counter()
time_row = t1 - t0

# 2. Sum along columns (strides across memory, causing cache misses)
t0 = time.perf_counter()
_ = np.sum(mat, axis=0)
t1 = time.perf_counter()
time_col = t1 - t0

print(f"Summing with memory flow (axis=1):    {time_row:.5f} s")
print(f"Summing against memory flow (axis=0): {time_col:.5f} s")

Summing with memory flow (axis=1):    0.05699 s
Summing against memory flow (axis=0): 0.00864 s


### 16.3 In-Place Operations to Save Memory
Expressions like `a = a + b` allocate a temporary intermediate array before reassigning.
Using in-place operators (`+=`, `*=`) or the `out=` parameter modifies data directly in RAM without any new allocation:

In [52]:
large_a = np.ones((1000, 1000), dtype=np.float64)
large_b = np.ones((1000, 1000), dtype=np.float64)

# Reassignment allocates a new 8 MB temporary buffer
id_before = id(large_a)
large_a = large_a + large_b
id_after_reassign = id(large_a)

# In-place operator modifies the exact same buffer!
large_a += large_b
id_after_inplace = id(large_a)

print("Buffer changed after reassign (+)?", id_before != id_after_reassign)
print("Buffer unchanged after inplace (+=)?", id_after_reassign == id_after_inplace)

Buffer changed after reassign (+)? True
Buffer unchanged after inplace (+=)? True


<a id="summary"></a>
---
# Part VI: Quick Reference & Practice Curriculum

## 🎯 Master NumPy Quick Reference Cheat Sheet

| Category | High-Frequency Functions |
| :--- | :--- |
| **Creation** | `np.array`, `np.zeros`, `np.ones`, `np.full`, `np.empty`, `np.arange`, `np.linspace`, `np.eye` |
| **Inspection** | `arr.shape`, `arr.ndim`, `arr.dtype`, `arr.size`, `arr.nbytes`, `arr.strides` |
| **Reshaping** | `arr.reshape(-1)`, `arr.ravel()`, `arr.flatten()`, `arr.T`, `np.squeeze`, `np.expand_dims` |
| **Stacking** | `np.vstack`, `np.hstack`, `np.concatenate(axis=...)`, `np.stack` |
| **Conditionals** | `np.where(cond, x, y)`, `arr[arr > 0]`, bitwise `&`, `|`, `~` |
| **Math & Ufuncs** | `np.exp`, `np.log`, `np.sqrt`, `np.clip`, `np.abs`, `np.round` |
| **Reductions** | `arr.sum(axis=...)`, `arr.mean()`, `arr.std()`, `arr.min()`, `arr.argmax()`, `keepdims=True` |
| **Linear Algebra** | `A @ B`, `np.linalg.inv`, `np.linalg.solve`, `np.linalg.det`, `np.linalg.norm`, `np.linalg.svd` |
| **Random (`rng`)** | `rng.random()`, `rng.integers()`, `rng.standard_normal()`, `rng.choice()`, `rng.shuffle()` |

---

### 🏋️ Ready to Test Your Skills?
Continue your mastery journey with our structured practice exercises in this directory:
1. **[02_numpy_exercise_1.ipynb](file:///Users/Inz_mac/Developer/ML/my-note/numpy/02_numpy_exercise_1.ipynb)**: 20 Hands-on exercises covering Array Creation, Slicing, and Reshaping.
2. **[03_numpy_exercise_1_solutions.ipynb](file:///Users/Inz_mac/Developer/ML/my-note/numpy/03_numpy_exercise_1_solutions.ipynb)**: Complete worked solutions for Exercise 1.
3. **[04_numpy_exercise_2.ipynb](file:///Users/Inz_mac/Developer/ML/my-note/numpy/04_numpy_exercise_2.ipynb)**: 15 Advanced problems covering Linear Algebra and ML Vectorization.
4. **[05_numpy_exercise_2_solutions.ipynb](file:///Users/Inz_mac/Developer/ML/my-note/numpy/05_numpy_exercise_2_solutions.ipynb)**: Detailed solutions and vectorization derivations for Exercise 2.

[⬆️ Back to Table of Contents](#table-of-contents)